# ScanNet HHA Phase 0 — Convention Validation + Drop List

**Purpose.** Before any HHA preprocessing run, validate that the math composition
`R_scannet = orthogonalize(axisAlignment[:3,:3]) @ pose[:3,:3]` produces a world frame
where +Z is gravity-up. Build a drop list of broken / drifted / high-NaN scenes that
`notebooks/scannet_preprocess.ipynb` reads to skip them at preprocessing time.

**Data flow** mirrors `scannet_preprocess.ipynb`: ScanNet raw is downloaded per-scene
on demand (the .txt metadata is a few KB each — batch downloaded — and the .sens
files are 100-200 MB each — downloaded one at a time, processed, then deleted).
Nothing persists to local disk; the drop list itself goes to Drive.

The output JSON lands at `/content/drive/MyDrive/datasets/scannet_drop_list.json`.

## Two-pass probe
- **T1 on ALL scenes** (cheap, .txt-only): orthogonality residual of the axisAlignment block.
- **Pass A (5 scenes, deep dive)**: download .sens; verify T0 (camera convention),
  T2 (composition direction), T4 (per-scene pose-drift distribution), T5 (depthShift).
- **Pass B (50 scenes, shallow)**: download .sens, sample a few frames, run the
  relative-floor test + NaN-rate gate + drift-frame fraction gate.


## 1. Install & Imports

In [ ]:
!pip install -q tqdm pypng

import glob
import hashlib
import json
import math
import os
import random
import re
import shutil
import struct
import subprocess
import sys
import time
import zlib
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm


## 2. Configuration

In [ ]:
TMP_DIR = '/content/scannet_tmp'
DRIVE = '/content/drive/MyDrive/datasets'
DROP_LIST_OUT = os.path.join(DRIVE, 'scannet_drop_list.json')

PASS_A_N = 5
PASS_B_N = 50            # Heavier than 200; each scene downloads ~150MB. Tune up if you want.
PASS_B_FRAMES_PER = 5    # Sample this many frames per scene for shallow checks.
SEED = 42

os.makedirs(TMP_DIR, exist_ok=True)
print(f'TMP: {TMP_DIR}')
print(f'Drive: {DRIVE}')


## 3. Mount Drive + Setup Tools

In [ ]:
from google.colab import drive
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    !fusermount -u /content/drive 2>/dev/null || true
    !rm -rf /content/drive
drive.mount('/content/drive')
os.makedirs(DRIVE, exist_ok=True)

TOOLS_DIR = '/content/scannet_tools'
os.makedirs(TOOLS_DIR, exist_ok=True)

# Download tools from ScanNet's official sources.
subprocess.run([
    'wget', '-q', '-O', os.path.join(TOOLS_DIR, 'download-scannet.py'),
    'http://kaldir.vc.cit.tum.de/scannet/download-scannet.py'
], check=True)
subprocess.run([
    'wget', '-q', '-O', os.path.join(TOOLS_DIR, 'SensorData.py'),
    'https://raw.githubusercontent.com/ScanNet/ScanNet/master/SensReader/python/SensorData.py'
], check=True)

# SensorData.py is Python 2 — patch for Python 3.
_sd_path = os.path.join(TOOLS_DIR, 'SensorData.py')
with open(_sd_path) as f:
    _lines = f.readlines()
_patched = 0
for _i, _line in enumerate(_lines):
    _orig = _line
    m = re.match(r'^(\s*)print (.+)$', _line.rstrip())
    if m and 'print(' not in _line:
        _lines[_i] = f'{m.group(1)}print({m.group(2)})\n'
    if '.join(struct.unpack(' in _line and "b'" not in _line.split('join')[0]:
        _lines[_i] = _lines[_i].replace("''.join(", "b''.join(")
    if 'np.fromstring' in _lines[_i]:
        _lines[_i] = _lines[_i].replace('np.fromstring', 'np.frombuffer')
    if _lines[_i] != _orig:
        _patched += 1
with open(_sd_path, 'w') as f:
    f.writelines(_lines)
assert _patched >= 7, f'SensorData.py patch failed: {_patched} lines'
print(f'SensorData.py patched ({_patched} lines)')

if TOOLS_DIR not in sys.path:
    sys.path.insert(0, TOOLS_DIR)

# Download official splits.
SPLITS_DIR = os.path.join(TOOLS_DIR, 'splits')
os.makedirs(SPLITS_DIR, exist_ok=True)
for f in ['scannetv2_train.txt', 'scannetv2_val.txt']:
    subprocess.run([
        'wget', '-q', '-O', os.path.join(SPLITS_DIR, f),
        f'https://raw.githubusercontent.com/ScanNet/ScanNet/master/Tasks/Benchmark/{f}'
    ], check=True)

def _load_ids(path):
    with open(path) as f:
        return [l.strip() for l in f if l.strip()]
train_scene_ids = _load_ids(os.path.join(SPLITS_DIR, 'scannetv2_train.txt'))
val_scene_ids   = _load_ids(os.path.join(SPLITS_DIR, 'scannetv2_val.txt'))
all_scene_ids = train_scene_ids + val_scene_ids
print(f'Total scenes: {len(all_scene_ids)} (train={len(train_scene_ids)}, val={len(val_scene_ids)})')

import SensorData  # noqa


## 4. Clone Repo (for compute_hha + scannet_intrinsics imports)

In [ ]:
from pathlib import Path

PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"

if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"Repo already exists: {LOCAL_REPO_PATH}")
    !cd {LOCAL_REPO_PATH} && git pull
else:
    if Path(LOCAL_REPO_PATH).exists():
        !rm -rf {LOCAL_REPO_PATH}
    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}

if LOCAL_REPO_PATH not in sys.path:
    sys.path.insert(0, LOCAL_REPO_PATH)

from src.data_utils.hha import (
    compute_hha,
    angular_distance_deg,
)
from src.data_utils.hha.scannet_intrinsics import _orthogonalize
print('Repo imports OK')


## 5. Download .txt Metadata for ALL Scenes (Lightweight)

Each .txt is a few KB. Total dataset: ~5 MB. We use these for T1
(orthogonality residual) on every scene without ever downloading a .sens.

In [ ]:
TXT_DIR = os.path.join(TMP_DIR, 'txt_metadata')
os.makedirs(TXT_DIR, exist_ok=True)

# Network-bound subprocess calls -> high thread fanout is fine.
TXT_DL_WORKERS = 32

def _dl_one_txt(scene_id):
    txt_out = os.path.join(TXT_DIR, 'scans', scene_id, f'{scene_id}.txt')
    if os.path.exists(txt_out):
        return ('skip', scene_id)
    cmd = ['python3', os.path.join(TOOLS_DIR, 'download-scannet.py'),
           '-o', TXT_DIR, '--id', scene_id, '--type', '.txt']
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=60, input='y\ny\n')
    return ('ok' if r.returncode == 0 else 'fail', scene_id)

print(f'Downloading .txt metadata for {len(all_scene_ids)} scenes ({TXT_DL_WORKERS} threads)...')
counts = Counter()
with ThreadPoolExecutor(max_workers=TXT_DL_WORKERS) as ex:
    futures = [ex.submit(_dl_one_txt, sid) for sid in all_scene_ids]
    for fut in tqdm(as_completed(futures), total=len(futures), desc='.txt'):
        status, _ = fut.result()
        counts[status] += 1
print(f"Downloaded: {counts['ok']}, Skipped (existing): {counts['skip']}, Failed: {counts['fail']}")

## 6. T1: Orthogonality Residuals + axisAlignment Availability (All Scenes)

For every scene, parse `<scene>.txt`:
  - If `axisAlignment` is missing → mark for the soft `missing_axisAlignment` reason
    (preprocess will fall back to identity, won't drop the scene).
  - If present, compute `||raw[:3,:3] - orthogonalized||_F` to flag scenes with
    unusually large scale folded in.

This is cheap — pure CPU work, no .sens downloads.

In [ ]:
def _parse_axisalignment(txt_path):
    if not os.path.isfile(txt_path):
        return None
    with open(txt_path) as f:
        for line in f:
            if line.startswith('axisAlignment'):
                m = re.match(r'axisAlignment\s*=\s*(.+)$', line)
                if m:
                    vals = np.fromstring(m.group(1), sep=' ', dtype=np.float64)
                    if vals.size == 16:
                        return vals.reshape(4, 4)
                break
    return None


t1_residuals = []  # list of (scene_id, residual)
missing_axis = []  # scenes with no axisAlignment field
for scene_id in tqdm(all_scene_ids, desc='T1 scan'):
    txt_path = os.path.join(TXT_DIR, 'scans', scene_id, f'{scene_id}.txt')
    A = _parse_axisalignment(txt_path)
    if A is None:
        missing_axis.append(scene_id)
        continue
    M = A[:3, :3]
    R = _orthogonalize(M)
    res = float(np.linalg.norm(M - R))
    t1_residuals.append((scene_id, res))

residuals = np.array([r for _, r in t1_residuals])
print(f'\nT1: {len(t1_residuals)} scenes have axisAlignment, {len(missing_axis)} missing')
if residuals.size:
    print(f'  residual: min={residuals.min():.4f}, max={residuals.max():.4f}, '
          f'p99={np.percentile(residuals, 99):.4f}')
    # Flag scenes with unusually large residuals (> p99 * 2 or > 0.5)
    large = [(sid, r) for sid, r in t1_residuals if r > max(0.5, np.percentile(residuals, 99) * 2)]
    print(f'  flagged (residual > thresh): {len(large)}')


## 7. Helper: Download a .sens, Open with SensorData, Delete When Done

This is the unit operation for Pass A and Pass B. Per-scene download takes
30-90 s on a typical Colab uplink; the .sens is 100-200 MB.

In [ ]:
def with_sens(scene_id, fn):
    """Download <scene_id>.sens, hand it to fn(sd), then delete the .sens.

    fn receives an opened SensorData.SensorData instance. Returns whatever fn returns.
    Returns None if download / open fails.
    """
    work_dir = os.path.join(TMP_DIR, 'sens_dl', scene_id)
    os.makedirs(work_dir, exist_ok=True)
    sens_path = os.path.join(work_dir, 'scans', scene_id, f'{scene_id}.sens')
    try:
        if not os.path.isfile(sens_path):
            cmd = ['python3', os.path.join(TOOLS_DIR, 'download-scannet.py'),
                   '-o', work_dir, '--id', scene_id, '--type', '.sens']
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=600, input='y\ny\n')
            if r.returncode != 0 or not os.path.isfile(sens_path):
                return None
        try:
            sd = SensorData.SensorData(sens_path)
        except Exception as e:
            print(f'  {scene_id}: SensorData open failed: {e}')
            return None
        return fn(sd)
    finally:
        shutil.rmtree(work_dir, ignore_errors=True)


def _backproject_camera(depth_m, K):
    H, W = depth_m.shape
    fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
    x = np.arange(1, W+1, dtype=np.float64)
    y = np.arange(1, H+1, dtype=np.float64)
    xx, yy = np.meshgrid(x, y)
    return np.stack([(xx-cx)*depth_m/fx, (yy-cy)*depth_m/fy, depth_m], axis=-1)


## 8. Pass A — 5-Scene Deep Dive (T0, T2, T4, T5)

For each scene: open .sens, grab K and depth_shift, sample 5 frames evenly.

- **T2 (composition direction).** Critical insight: ScanNet's `axisAlignment` is by
  construction a rotation around the gravity axis (Z), so its third row is
  `[0, 0, ±1]`. That means `R_align @ pose` and `R_align.T @ pose` have identical
  third rows, and HHA channels (height + angle-with-gravity) depend only on world-Z.
  So **the choice of `axis_alignment_inverted` cannot affect HHA on ScanNet.** We
  verify this Z-only property directly and report the floor-cluster fraction
  (fraction of points within 15 cm of the 5th-percentile world-Z) for the canonical,
  inverted, and no-alignment hypotheses as a sanity check.
- **T4 (pose validity).** Per-frame: pose is finite, `det(R) ≈ 1`, `R^T R ≈ I`.
  We DO NOT measure inter-frame angular drift — that's normal for handheld scans.
- **T5 (depth shift).** Should be 1000 mm/unit for all ScanNet scenes.

In [ ]:
rng = np.random.default_rng(SEED)

# Pick Pass A from scenes that have axisAlignment + are in the splits.
candidates_a = [sid for sid, _ in t1_residuals]
pass_a_scenes = list(rng.choice(candidates_a, size=min(PASS_A_N, len(candidates_a)), replace=False))
print(f'Pass A scenes: {pass_a_scenes}\n')


def _is_rotation_valid(R, tol=1e-3):
    """True iff R is finite, det(R) ≈ +1, and R^T R ≈ I."""
    if not np.all(np.isfinite(R)):
        return False
    if abs(np.linalg.det(R) - 1.0) > tol:
        return False
    if np.linalg.norm(R.T @ R - np.eye(3)) > tol:
        return False
    return True


def _floor_cluster_metrics(pts_world_valid):
    """Return (z_spread_m, near_floor_frac).
    near_floor_frac = fraction of points within 0.15 m of the 5th-percentile world-Z.
    Higher value = points cluster on a horizontal plane = +Z is gravity-up.
    """
    z = pts_world_valid[..., 2]
    if z.size < 100:
        return None, None
    z_lo, z_hi = np.percentile(z, [1, 99])
    spread = float(z_hi - z_lo)
    floor_z = float(np.percentile(z, 5))
    near_floor = float(np.mean(np.abs(z - floor_z) < 0.15))
    return spread, near_floor


def pass_a_per_scene(sd, scene_id, axis_align_4x4):
    K = np.asarray(sd.intrinsic_depth, dtype=np.float64)[:3, :3]
    depth_shift = float(getattr(sd, 'depth_shift', 1000.0))
    R_align = _orthogonalize(axis_align_4x4[:3, :3])

    # Z-only check: pure Z-rotation has third row = [0, 0, ±1].
    third_row = R_align[2, :]
    z_dominance = float(abs(third_row[2]) - np.linalg.norm(third_row[:2]))
    is_z_rotation = bool(z_dominance > 0.99)

    # Pose validity across 5 sampled frames.
    n = len(sd.frames)
    if n < 5:
        return None
    sample_idxs = list(np.linspace(0, n - 1, 5, dtype=int))
    n_invalid = 0
    for fi in sample_idxs:
        pose = np.asarray(sd.frames[fi].camera_to_world, dtype=np.float64)
        if pose.shape != (4, 4) or not _is_rotation_valid(pose[:3, :3]):
            n_invalid += 1

    # Median frame: backproject + measure floor clustering for 3 hypotheses.
    fi = sample_idxs[len(sample_idxs) // 2]
    try:
        depth_data = sd.frames[fi].decompress_depth(sd.depth_compression_type)
        depth_mm = np.frombuffer(depth_data, dtype=np.uint16).reshape(sd.depth_height, sd.depth_width)
    except Exception:
        return None
    depth_m = depth_mm.astype(np.float32) / depth_shift
    valid = depth_m > 0
    if valid.sum() < 1000:
        return None
    pose = np.asarray(sd.frames[fi].camera_to_world, dtype=np.float64)
    if not _is_rotation_valid(pose[:3, :3]):
        return None
    pts_cam = _backproject_camera(depth_m, K)
    pts_cam_v = pts_cam[valid]

    hypotheses = {
        'R_align @ pose  (canonical)':  R_align        @ pose[:3, :3],
        'R_align.T @ pose (inverted)': R_align.T      @ pose[:3, :3],
        'pose only       (no align)':  pose[:3, :3],
    }
    floor_scores = {}
    for label, R in hypotheses.items():
        pts_world_v = pts_cam_v @ R.T
        spread, near = _floor_cluster_metrics(pts_world_v)
        floor_scores[label] = (spread, near)

    return {
        'scene_id': scene_id,
        'depth_shift': depth_shift,
        'is_z_rotation': is_z_rotation,
        'z_dominance': z_dominance,
        'n_invalid_poses_of_5': n_invalid,
        'floor_scores': floor_scores,
        'native_depth_hw': (sd.depth_height, sd.depth_width),
    }


pass_a_results = []

def _pass_a_worker(sid):
    txt = os.path.join(TXT_DIR, 'scans', sid, f'{sid}.txt')
    A = _parse_axisalignment(txt)
    if A is None:
        return (sid, 'missing', None)
    res = with_sens(sid, lambda sd: pass_a_per_scene(sd, sid, A))
    return (sid, None, res)

# 5 scenes, ~150 MB .sens each -> 5-way parallel download.
PASS_A_WORKERS = min(5, len(pass_a_scenes))
results_by_sid = {}
with ThreadPoolExecutor(max_workers=PASS_A_WORKERS) as ex:
    futs = {ex.submit(_pass_a_worker, sid): sid for sid in pass_a_scenes}
    for fut in tqdm(as_completed(futs), total=len(futs), desc='Pass A'):
        sid, flag, res = fut.result()
        results_by_sid[sid] = (flag, res)

# Print in submission order so output is deterministic.
for sid in pass_a_scenes:
    flag, res = results_by_sid.get(sid, (None, None))
    if flag == 'missing':
        print(f'{sid}: missing axisAlignment, skipping Pass A')
        continue
    print(f'\n--- {sid} ---')
    if res is None:
        print('  (no result)')
        continue
    pass_a_results.append(res)
    print(f'  depth_shift: {res["depth_shift"]} (expect 1000)')
    print(f'  R_align is Z-only rotation: {res["is_z_rotation"]} '
          f'(z_dominance={res["z_dominance"]:+.3f}; >0.99 means pure Z-rot)')
    print(f'  invalid poses (of 5 sampled): {res["n_invalid_poses_of_5"]}')
    print(f'  floor-cluster (spread_m, near_floor_frac):')
    for label, (spread, near) in res['floor_scores'].items():
        spread_s = f'{spread:5.2f}' if spread is not None else '  n/a'
        near_s   = f'{near:.3f}'    if near   is not None else 'n/a'
        print(f'    {label:32s}  spread={spread_s} m  near_floor={near_s}')
    print(f'  native depth: {res["native_depth_hw"]}')

# --- T2 verdict ---
all_z_rot = all(r['is_z_rotation'] for r in pass_a_results)
print(f'\nT2 verdict:')
print(f'  All Pass A scenes have Z-only axisAlignment: {all_z_rot}')
if all_z_rot and pass_a_results:
    print('  -> R_align preserves world-Z. axis_alignment_inverted is irrelevant for HHA.')
    print('  -> Defaulting axis_alignment_inverted = False.')
    AXIS_ALIGNMENT_INVERTED = False
else:
    # Fallback: pick the hypothesis with the higher mean near_floor.
    def _mean_near(label):
        vals = [r['floor_scores'][label][1] for r in pass_a_results
                if r['floor_scores'][label][1] is not None]
        return float(np.mean(vals)) if vals else 0.0
    canon = _mean_near('R_align @ pose  (canonical)')
    inv   = _mean_near('R_align.T @ pose (inverted)')
    AXIS_ALIGNMENT_INVERTED = inv > canon
    print(f'  Mean near-floor: canonical={canon:.3f}, inverted={inv:.3f}')
    print(f'  -> axis_alignment_inverted = {AXIS_ALIGNMENT_INVERTED}')

# T5 verdict
unique_shifts = sorted({r['depth_shift'] for r in pass_a_results})
print(f'T5 unique depth_shifts: {unique_shifts}')

# Sanity: chosen hypothesis should give a non-trivial floor cluster.
chosen_label = 'R_align.T @ pose (inverted)' if AXIS_ALIGNMENT_INVERTED else 'R_align @ pose  (canonical)'
chosen = [r['floor_scores'][chosen_label][1] for r in pass_a_results
          if r['floor_scores'][chosen_label][1] is not None]
if chosen:
    print(f'Chosen hypothesis ({chosen_label.strip()}):')
    print(f'  near_floor mean={np.mean(chosen):.3f}  min={min(chosen):.3f}  max={max(chosen):.3f}')

## 9. Pass B — 50-Scene Shallow Scan (NaN gate, pose-validity gate, floor-cluster gate)

For each scene: download .sens, sample `PASS_B_FRAMES_PER` frames evenly, then drop
the scene if ANY of these are true:

1. **Mean depth NaN rate > 30%** — sensor saturation / large invalid regions.
2. **Invalid-pose fraction > 50%** — `pose` is non-finite, `det(R) ≠ 1`, or
   `R^T R ≠ I`. Real tracking failure.
3. **Mean floor-cluster fraction < 5%** — after composing `R_align @ pose`, points
   should cluster on a horizontal plane in world-Z. If they don't, the world frame
   is not gravity-up for that scene.

We DO NOT drop scenes for inter-frame rotational "drift" — that's expected behavior
for a handheld scan where the operator rotates the camera through a room.

In [ ]:
# Pick Pass B scenes (overlap with Pass A is fine — small).
pass_b_scenes = list(rng.choice(candidates_a, size=min(PASS_B_N, len(candidates_a)), replace=False))
print(f'Pass B: {len(pass_b_scenes)} scenes ({PASS_B_FRAMES_PER} frames each)\n')


def pass_b_per_scene(sd, scene_id, axis_align_4x4, axis_inverted):
    raw = axis_align_4x4[:3, :3]
    if axis_inverted:
        raw = raw.T
    R_align = _orthogonalize(raw)
    depth_shift = float(getattr(sd, 'depth_shift', 1000.0))
    K = np.asarray(sd.intrinsic_depth, dtype=np.float64)[:3, :3]

    n = len(sd.frames)
    if n < 5:
        return None
    sample_idxs = list(np.linspace(0, n - 1, PASS_B_FRAMES_PER, dtype=int))
    nan_rates = []
    floor_scores = []
    invalid_pose_count = 0
    sampled_count = 0

    for fi in sample_idxs:
        sampled_count += 1
        pose = np.asarray(sd.frames[fi].camera_to_world, dtype=np.float64)
        if pose.shape != (4, 4) or not _is_rotation_valid(pose[:3, :3]):
            invalid_pose_count += 1
            continue
        try:
            depth_data = sd.frames[fi].decompress_depth(sd.depth_compression_type)
            depth_mm = np.frombuffer(depth_data, dtype=np.uint16).reshape(sd.depth_height, sd.depth_width)
        except Exception:
            continue
        depth_m = depth_mm.astype(np.float32) / depth_shift
        valid = depth_m > 0
        nan_rates.append(float(1.0 - valid.mean()))
        if valid.sum() < 1000:
            continue
        pts_cam = _backproject_camera(depth_m, K)
        pts_world_v = pts_cam[valid] @ (R_align @ pose[:3, :3]).T
        _, near = _floor_cluster_metrics(pts_world_v)
        if near is not None:
            floor_scores.append(near)

    invalid_frac = float(invalid_pose_count / max(sampled_count, 1))
    return {
        'scene_id': scene_id,
        'mean_nan_rate':       float(np.mean(nan_rates))    if nan_rates    else None,
        'mean_near_floor':     float(np.mean(floor_scores)) if floor_scores else None,
        'invalid_pose_frac':   invalid_frac,
    }


drop_entries = {}
nan_rates_all = []
near_floor_all = []
invalid_pose_all = []

# Soft-flag all 'missing axisAlignment' scenes regardless of Pass B sample.
for sid in missing_axis:
    drop_entries[sid] = 'missing_axisAlignment'


def _pass_b_worker(sid):
    txt = os.path.join(TXT_DIR, 'scans', sid, f'{sid}.txt')
    A = _parse_axisalignment(txt)
    if A is None:
        return (sid, 'missing', None)
    res = with_sens(sid, lambda sd: pass_b_per_scene(sd, sid, A, AXIS_ALIGNMENT_INVERTED))
    return (sid, None, res)

# 50 scenes, ~150 MB .sens each. 4 in flight = ~600 MB peak disk.
PASS_B_WORKERS = 4
with ThreadPoolExecutor(max_workers=PASS_B_WORKERS) as ex:
    futs = {ex.submit(_pass_b_worker, sid): sid for sid in pass_b_scenes}
    for fut in tqdm(as_completed(futs), total=len(futs), desc='Pass B'):
        sid, flag, res = fut.result()
        if flag == 'missing':
            drop_entries[sid] = 'missing_axisAlignment'
            continue
        if res is None:
            continue

        if res['mean_nan_rate'] is not None:
            nan_rates_all.append(res['mean_nan_rate'])
        if res['mean_near_floor'] is not None:
            near_floor_all.append(res['mean_near_floor'])
        invalid_pose_all.append(res['invalid_pose_frac'])

        # Hard-drop gates: real defects only.
        if res['mean_nan_rate'] is not None and res['mean_nan_rate'] > 0.30:
            drop_entries[sid] = f"high_nan_rate={res['mean_nan_rate']:.2f}"
            continue
        if res['invalid_pose_frac'] > 0.50:
            drop_entries[sid] = f"invalid_poses_{int(res['invalid_pose_frac']*100)}pct"
            continue
        if res['mean_near_floor'] is not None and res['mean_near_floor'] < 0.05:
            drop_entries[sid] = f"weak_floor_cluster={res['mean_near_floor']:.3f}"

print(f'\nPass B summary:')
print(f'  Scenes scanned: {len(pass_b_scenes)}')
print(f'  Soft-flagged missing_axisAlignment: {len(missing_axis)}')
print(f'  Hard-dropped (broken / high-nan / invalid-pose / weak-floor): '
      f'{sum(1 for v in drop_entries.values() if not v.startswith("missing_"))}')
if nan_rates_all:
    print(f'  Mean NaN rate: {np.mean(nan_rates_all):.3f}, p99: {np.percentile(nan_rates_all, 99):.3f}')
if invalid_pose_all:
    print(f'  Mean invalid-pose fraction: {np.mean(invalid_pose_all):.3f}')
if near_floor_all:
    print(f'  Mean near-floor cluster: {np.mean(near_floor_all):.3f}')

## 10. Save Drop List to `/content/drive/MyDrive/datasets/scannet_drop_list.json`

In [ ]:
out = {
    'convention_verified': True,
    'axis_alignment_inverted': bool(AXIS_ALIGNMENT_INVERTED),
    'scenes': drop_entries,
    'phase0_metadata': {
        'pass_a_n': PASS_A_N,
        'pass_b_n': PASS_B_N,
        'pass_b_frames_per_scene': PASS_B_FRAMES_PER,
        'seed': SEED,
        't1_total_scenes': len(t1_residuals),
        't1_residual_max': float(residuals.max()) if residuals.size else None,
        't5_unique_depth_shifts': unique_shifts,
        'mean_nan_rate':              float(np.mean(nan_rates_all))    if nan_rates_all    else None,
        'mean_invalid_pose_fraction': float(np.mean(invalid_pose_all)) if invalid_pose_all else None,
        'mean_near_floor':            float(np.mean(near_floor_all))   if near_floor_all   else None,
    },
}
with open(DROP_LIST_OUT, 'w') as f:
    json.dump(out, f, indent=2)
print(f'Wrote {DROP_LIST_OUT}')
print(f'  axis_alignment_inverted: {AXIS_ALIGNMENT_INVERTED}')
print(f'  total scenes flagged: {len(drop_entries)}')
print(f'    soft (missing_axisAlignment, identity fallback): '
      f'{sum(1 for v in drop_entries.values() if v.startswith("missing_"))}')
print(f'    hard-dropped (broken/high-nan/invalid-pose/weak-floor): '
      f'{sum(1 for v in drop_entries.values() if not v.startswith("missing_"))}')